<a href="https://colab.research.google.com/github/varba187/RAGs-to-Riches/blob/main/code/notebooks/bart_baseline_train_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install transformers datasets sentencepiece accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.6 MB/s eta 0:00:00


In [3]:
import re
import torch
import pandas as pd

from datasets import load_dataset
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(F"Device set to {device}")

torch.manual_seed(0)

Device set to cpu


# Load NQ

In [4]:
dataset = load_dataset("sentence-transformers/natural-questions", split="train")
print(dataset)
print(dataset.column_names)
print(dataset[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

pair/train-00000-of-00001.parquet:   0%|          | 0.00/44.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100231 [00:00<?, ? examples/s]

Dataset({
    features: ['query', 'answer'],
    num_rows: 100231
})
['query', 'answer']
{'query': 'when did richmond last play in a preliminary final', 'answer': "Richmond Football Club Richmond began 2017 with 5 straight wins, a feat it had not achieved since 1995. A series of close losses hampered the Tigers throughout the middle of the season, including a 5-point loss to the Western Bulldogs, 2-point loss to Fremantle, and a 3-point loss to the Giants. Richmond ended the season strongly with convincing victories over Fremantle and St Kilda in the final two rounds, elevating the club to 3rd on the ladder. Richmond's first final of the season against the Cats at the MCG attracted a record qualifying final crowd of 95,028; the Tigers won by 51 points. Having advanced to the first preliminary finals for the first time since 2001, Richmond defeated Greater Western Sydney by 36 points in front of a crowd of 94,258 to progress to the Grand Final against Adelaide, their first Grand Final a

# First, let's train on a small dataset

In [5]:
small_dataset = dataset.select(range(5000))
split_dataset = small_dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

In [6]:
model_name = "facebook/bart-large"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)
model = model.to(device)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/513 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


# Data preprocessing

In [7]:
max_input_length = 128
max_output_length = 32

def get_inputs(dataset_entry):
  question = dataset_entry["query"]
  answer = dataset_entry["answer"]

  if (isinstance(answer, list)):
    answer = answer[0]

  model_inputs = tokenizer(question, max_length = max_input_length, truncation = True)
  labels = tokenizer(text_target = answer, max_length = max_output_length, truncation = True)
  model_inputs["labels"] = labels["input_ids"]

  return model_inputs

In [8]:
tokenized_train = train_dataset.map(get_inputs, remove_columns=train_dataset.column_names)
tokenized_test = test_dataset.map(get_inputs, remove_columns=test_dataset.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

# Here we define some important arguments for training

In [9]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=1,
    logging_strategy="steps",
    logging_steps=100,
    predict_with_generate=True,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    save_total_limit=1,
    report_to="none"
)

# Normalization

In [10]:
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

# Exact Match Metrics

In [11]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    decoded_pred = tokenizer.batch_decode(predictions, skip_special_tokens = True)
    labels = [[token if token != -100 else tokenizer.pad_token_id for token in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens = True)

    em = [int(normalize_text(pred) == normalize_text(label)) for pred, label in zip(decoded_pred, decoded_labels)]

    return {"em": 100 * sum(em) / len(em)}

# Training

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss


In [ ]:
metrics = trainer.evaluate()
print(metrics)

# Predictions generation

In [ ]:
predictions = trainer.predict(tokenized_test)
print(predictions.predictions.shape, predictions.label_ids.shape)

In [ ]:
preds, labels = predictions.predictions, predictions.label_ids
decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

labels = [[token if token != -100 else tokenizer.pad_token_id for token in label] for label in labels]
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

results_df = pd.DataFrame({"question": test_dataset["query"], "answer": decoded_labels, "prediction": decoded_preds})
results_df["em"] = [int(normalize_text(prediction) == normalize_text(result)) for prediction, result in zip(results_df["prediction"], results_df["answer"])]

In [ ]:
results_df.to_csv('bart_baseline_train_eval_results.csv', index=False)
print("Results saved to bart_baseline_train_eval_results.csv")
results_df.head()